In [22]:
import anthropic

In [23]:
from dotenv import load_dotenv
import anthropic

load_dotenv(".env")  # .env 파일 읽기

client = anthropic.Anthropic()  # 대문자 A, 괄호 안에 아무것도 안 넣어도 됨

In [24]:
import os
print(os.getenv("ANTHROPIC_API_KEY"))

YOUR_API_KEY_HERE


In [25]:
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": "안녕! 너는 Tool Use가 뭔지 중학생도 이해할 수 있게 설명해줄 수 있겠니?"
        }
    ]
)

In [26]:
from IPython.display import Markdown
display(Markdown(response.content[0].text))

# Tool Use 쉽게 설명하기 🛠️

## 먼저 비유로 이해해보자!

> 🧑‍🍳 **요리사** 비유를 들어볼게요!

요리사가 음식을 만들 때 **손만 쓰지 않잖아요?**
- 칼 🔪 → 재료를 자를 때
- 냄비 🍳 → 끓일 때
- 저울 ⚖️ → 양을 잴 때

**도구를 상황에 맞게 골라서 씁니다!**

---

## AI에서 Tool Use란?

AI도 똑같아요!

AI 혼자서는 못하는 일들이 있어요.

| 못하는 것 | 도구를 쓰면? |
|---|---|
| 오늘 날씨 모름 | 날씨 검색 도구 사용 🌤️ |
| 계산이 복잡함 | 계산기 도구 사용 🔢 |
| 최신 뉴스 모름 | 인터넷 검색 도구 사용 🔍 |

---

## 한 문장 정리! ✅

> **"AI가 스스로 판단해서, 필요한 도구를 직접 골라 쓰는 것"**

---

## 실제 예시 🙋

> "오늘 서울 날씨 알려줘!"

1. AI가 생각함 → *"나는 오늘 날씨를 모르네?"*
2. AI가 선택함 → *"날씨 검색 도구를 써야겠다!"*
3. 도구 사용 → 검색 실행!
4. 결과 전달 → "오늘 서울은 맑고 23도예요 ☀️"

---

궁금한 점 있으면 더 물어봐요! 😊

In [27]:
# step2: Tool use 구현하기 
# 목표 : claude가 "도구"를 사용하는 에이전트 만들기


# 🧠 Tool Use가 뭐냐면:
#    평소 Claude는 "말"만 할 수 있잖아요?
#    Tool Use는 Claude에게 "도구 목록"을 알려주면,
#    Claude가 스스로 판단해서 필요한 도구를 "호출"하는 겁니다.

In [28]:
import anthropic
import json

client = anthropic.Anthropic()


# json Schema 형식으로 도구의 이름, 설명, 파라미터를 정의
tools = [
    {
        "name": "get_weather",
        "description": "특정 도시의 현재 날씨 정보를 가져옵니다.사용자가 날씨를 물어볼 때 사용하세요.",
        "input_schema": {
            "city": {
                "type": "stirng",
                "description": "날씨를 조회할 도시 이름 (예: 서울, 청주, 대구)"
            }
        },
        "required": ["city"]
    },
    {
        "name": "get_time",
        "description": "특정 도시의 현재 시간을 가져옵니다.",
        "input_schema": {
            "type": "object",
            "property": {
                "city": {
                    "type": "string",
                    "description": "시간을 조회할 도시 이름"
                }
            },
            "required": ["city"]
        }
        
    }
]



In [29]:
# ============================================================
# 📌 PART 2: 도구가 실제로 하는 일 (가짜 데이터)
# ============================================================
# 실제 서비스라면 여기서 API를 호출하겠지만,
# 학습용이니까 가짜 데이터를 돌려줍니다.
# 나중에 여기를 진짜 API로 바꾸면 실전 에이전트가 됩니다!


def get_weather(city: str) -> dict:
    """가짜 날씨 데이터 반환 (나중에 진짜 API로 교체 가능)"""
    fake_weather = {
        "서울": {"temp": 3, "condition": "맑음", "humidity": 45},
        "부산": {"temp": 7, "condition": "흐림", "humidity": 60},
        "뉴욕": {"temp": -2, "condition": "눈", "humidity": 70},
    }
    weather = fake_weather.get(city, {"temp": 20, "condition": "알 수 없음", "humidity": 50})
    return {"city": city, **weather}


def get_time(city: str) -> dict:
    """가짜 시간 데이터 반환"""
    fake_time = {
        "서울": "2025-02-23 05:45:00 KST",
        "부산": "2025-02-23 05:45:00 KST",
        "뉴욕": "2025-02-22 15:45:00 EST", 
    }
    return {"city": city, "current_time": fake_time.get(city, "알 수 없음")}

# 도구 이름 → 실제 함수 매핑 (claude가 호출되면 이 딕셔너리에서 찾아서 실행)
tool_functions = {
    "get_weather": get_weather,
    "get_time": get_time,
}




In [30]:
# ============================================================
# 📌 PART 3: Tool Use 대화 루프 (핵심!)
# ============================================================
# 이 함수가 에이전트의 "두뇌"입니다.
# 
# 흐름:
# 1) 사용자 메시지를 Claude에게 보냄 (도구 목록도 같이)
# 2) Claude가 "도구를 쓰겠다"고 응답하면 → 우리가 실제로 실행
# 3) 실행 결과를 다시 Claude에게 보냄
# 4) Claude가 최종 답변을 만들어줌
#
# 이 루프가 바로 ReAct 패턴의 기초입니다!

def run_agent(user_message: str):
    """Tool Use 에이전트 실행"""
    
    print(f"\n{'='*60}")
    print(f"👤 사용자: {user_message}")
    print(f"{'='*60}")
    
    messages = [{"role": "user", "content": user_message}]
    
    # ---- 1단계: Claude에게 메시지 + 도구 목록 보내기 ----
    response = client. messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        tools=tools,
        messages=messages,
    )
    
    print(f"\n🤖 Claude의 판단: stop_reason = '{response.stop_reason}'")
    
    # ---- 5단계: 최종 응답 출력 ----
    final_text = ""
    for block in response.content:
        if hasattr(block, "text"):
            final_text += block.text
        
    print(f"\n💬 Claude 최종 답변:\n{final_text}")
    print(f"\n{'='*60}\n")

    return final_text


In [ ]:
# ============================================================
# 📌 PART 4: 테스트해보기!
# ============================================================
# 다양한 질문으로 Claude가 어떻게 도구를 쓰는지 관찰하세요

if __name__ == "__main__":
    
    # 테스트 1: 날씨 질문 → Claude가 get_weather를 호출할 것
    run_agent("서울 날씨 어때?")
    
    # 테스트 2: 시간 질문 → Claude가 get_time을 호출할 것
    run_agent("뉴욕은 지금 몇 시야?")
    
    # 테스트 3: 도구가 필요 없는 질문 → Claude가 도구 없이 답변할 것
    run_agent("파이썬이 뭐야?")
    
    # 테스트 4: 복합 질문 → Claude가 여러 도구를 쓸 수도 있음
    run_agent("서울 날씨랑 뉴욕 시간 알려줘")


👤 사용자: 서울 날씨 어때?


BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'tools.0.custom.input_schema.type: Field required'}, 'request_id': 'req_011CYrLWwid4ozxEF5WdnQyA'}

: 